# Spectral models

In [ ]:
import astropy.units as u
from astropy.units import Quantity
import numpy as np
from abc import ABC, abstractmethod
from scipy.integrate import quad
import matplotlib.pyplot as plt

In [ ]:
plt.rc('font'  , size=28)       # Controls default text sizes
plt.rc('figure', titlesize=40)  # Fontsize of the figure suptitle
plt.rc('axes'  , titlesize=40)  # Fontsize of the title (ax.set_title)

plt.rc('axes'  , labelsize=30)  # Fontsize of the x and y labels
plt.rc('xtick' , labelsize=20)  # Fontsize of the tick labels
plt.rc('ytick' , labelsize=20)  # Fontsize of the tick labels
plt.rc('legend', title_fontsize=30)  # Legend fontsize title
plt.rc('legend', fontsize=16)   # Legend fontsize text

plt.rc('lines' ,markersize=14)  # Markersize

plt.rc('font', family="sans-serif", style="normal", variant="normal",weight="normal")
# plt.rc('font'  , family="Times New Roman", style="normal", variant="normal",weight="normal")

FIGSIZE=(16,12)


## Class Definitions

#### Base Spectral Model

In [ ]:
###################################################################################
class SpectralModel(ABC):
    """
    Base class to deal with spectral model conversions.
    """
    unit_energy = u.Unit("keV")
    unit_amplitude = u.Unit("s-1 cm-2 keV-1")
    unit_flux = u.Unit("s-1 cm-2")
    unit_SED = u.Unit("erg s-1 cm-2")
    
    def __init__(self):
        self.N0 = 0.0 * SpectralModel.unit_amplitude
        self.E0 = 1.0 * SpectralModel.unit_energy

    def set_N0(self, N0 : Quantity):
        if not N0.unit.is_equivalent(SpectralModel.unit_amplitude):
            raise ValueError
        self.N0 = N0

    def set_E0(self, E0 : Quantity):
        if not E0.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        self.E0 = E0

    def get_N0(self):
        return self.N0
    
    def get_E0(self):
        return self.E0
    
    def __str__(self):
        return super().__str__()
    
    def __repr__(self):
        return self.__str__()
    
    @abstractmethod
    def _spectral_model(self, energy_normalised):
        """
        Abstract Method to define the spectral models.
        It must return an adimensional value.
        
        Parameters
        ----------
        energy_normalised : `np.array`
            Energy where to compute the model, expressed as adimensional energy / E0.
        """
        pass

    def _E_spectral_model(self, energy_normalised):
        return energy_normalised * self._spectral_model(energy_normalised)
    
    def get_dNdE(self, energy : Quantity):
        """Evaluate dNdE at a given energy."""
        if not energy.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        
        energy_normalised = (energy/self.get_E0()).to('')
        dNdE = self.N0 * self._spectral_model(energy_normalised)
        return dNdE.to(SpectralModel.unit_amplitude)
    
    
    def get_E2dNdE(self, energy : Quantity):
        """Evaluate E2dNdE at a given energy."""
        if not energy.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        
        E2dNdE = np.power(energy,2) * self.get_dNdE(energy)
        return E2dNdE.to(SpectralModel.unit_SED)
    

    def get_flux(self, E1 : Quantity, E2 : Quantity, return_error=False):
        """Evaluate Photon flux Between Two Energies"""
        if not E1.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        if not E2.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        
        E1_normalised = (E1/self.E0).to('')
        E2_normalised = (E2/self.E0).to('')
        
        flux, error = self.N0 * self.E0 * quad(self._spectral_model, E1_normalised, E2_normalised)

        if return_error:
            return flux.to(SpectralModel.unit_flux), error.to(SpectralModel.unit_flux)
        return flux.to(SpectralModel.unit_flux)
    


    def get_eflux(self, E1 : Quantity, E2 : Quantity, return_error=False):
        """Evaluate Photon flux Between Two Energies"""
        if not E1.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        if not E2.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        
        E1_normalised = (E1/self.E0).to('')
        E2_normalised = (E2/self.E0).to('')
        
        eflux, error = self.N0 * np.power(self.E0,2) * quad(self._E_spectral_model, E1_normalised, E2_normalised)

        if return_error:
            return eflux.to(SpectralModel.unit_SED), error.to(SpectralModel.unit_SED)
        return eflux.to(SpectralModel.unit_SED)
    


    def estimate_N0_from_flux(self, E1 : Quantity, E2 : Quantity, flux : Quantity):
        """Estimate the Amplitude from the flux"""
        if not E1.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        if not E2.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        if not flux.unit.is_equivalent(SpectralModel.unit_flux):
            raise ValueError
        
        E1_normalised = (E1/self.E0).to('')
        E2_normalised = (E2/self.E0).to('')
        
        integral, error = quad(self._spectral_model, E1_normalised, E2_normalised)

        N0 = flux / (self.E0 * integral)

        return N0.to(SpectralModel.unit_amplitude)



#### Power Law

In [ ]:
class PowerLawModel(SpectralModel):
    
    def __init__(self, index : float):
        """Set the Index. Amplitude must be set with the appropriate method"""
        super().__init__()
        self.index = float(index)

    def _spectral_model(self, energy_normalised):
        """Evaluate the spectral model"""
        return np.power(energy_normalised,-self.index)
    
    def __str__(self):
        return f"Power Law Model\n N0={self.N0:.3e}\n E0={self.E0}\n Index={self.index}\n"
    

    def get_flux(self, E1 : Quantity, E2 : Quantity):
        """Redefine to use analytical expressions"""
        if not E1.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        if not E2.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        
        E1_normalised = (E1/self.E0).to('')
        E2_normalised = (E2/self.E0).to('')
        
        if self.index == 1.0:
            integral = np.log(E2_normalised / E1_normalised)
        else:
            integral = np.power(E2_normalised, 1-self.index)-np.power(E1_normalised, 1-self.index) / (1-self.index)
        
        flux = self.N0 * self.E0 * integral

        return flux.to(SpectralModel.unit_flux)
    

    def get_eflux(self, E1 : Quantity, E2 : Quantity):
        """Evaluate Photon flux Between Two Energies"""
        if not E1.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        if not E2.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        
        E1_normalised = (E1/self.E0).to('')
        E2_normalised = (E2/self.E0).to('')
        
        if self.index == 2.0:
            integral = np.log(E2_normalised / E1_normalised)
        else:
            integral = np.power(E2_normalised, 2-self.index)-np.power(E1_normalised, 2-self.index) / (2-self.index)
        
        eflux = self.N0 * np.power(self.E0,2) * integral

        return eflux.to(SpectralModel.unit_SED)
    
    def estimate_N0_from_flux(self, E1 : Quantity, E2 : Quantity, flux : Quantity):
        """Redefine to use analytical expressions"""
        if not E1.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        if not E2.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        if not flux.unit.is_equivalent(SpectralModel.unit_flux):
            raise ValueError
        
        E1_normalised = (E1/self.E0).to('')
        E2_normalised = (E2/self.E0).to('')
        
        if self.index == 1.0:
            integral = np.log(E2_normalised / E1_normalised)
        else:
            integral = np.power(E2_normalised, 1-self.index)-np.power(E1_normalised, 1-self.index) / (1-self.index)
        
        N0 = flux / (self.E0 * integral)

        return N0.to(SpectralModel.unit_amplitude)

#### Band

In [ ]:
class BandModel(SpectralModel):
    
    def __init__(self, lower_index : float, upper_index : float, E_break : Quantity):
        """Set the Index. Amplitude must be set with the appropriate method"""
        super().__init__()
        if not E_break.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        if lower_index - upper_index <=0:
            raise ValueError
        
        self.lower_index = float(lower_index)
        self.upper_index = float(upper_index)
        self.E_break = E_break

    def E_transition(self):
        """Transition Energy"""
        return (self.lower_index-self.upper_index)*self.E_break

    def _spectral_model(self, energy_normalised):
        """Evaluate the spectral model"""
        E_transition_normalised = (self.E_transition()/self.E0).to('')
        E_break_normalised = (self.E_break/self.E0).to('')

        value = np.where(energy_normalised <=  E_transition_normalised,
                         np.power(energy_normalised, self.lower_index) * np.exp(- energy_normalised / E_break_normalised),
                         np.power(energy_normalised, self.upper_index) * np.power(E_break_normalised*(self.lower_index-self.upper_index), self.lower_index-self.upper_index) * np.exp(self.upper_index-self.lower_index)
                         )
        return value
    
    def __str__(self):
        string = f"Band Model\n N0={self.N0:.3e}\n E0={self.E0}\n"
        string+= f" Lower Index={self.lower_index}\n Upper Index={self.upper_index}\n"
        string+= f" Break Energy={self.E_break}\n Transition Energy={self.E_transition():.3e}\n"
        return string



#### Comptonized

In [ ]:
class Comptonized(SpectralModel):
    
    def __init__(self, index : float, E_peak : Quantity):
        """Set the Index and Peak Energy. Amplitude must be set with the appropriate method"""
        super().__init__()
        if not E_peak.unit.is_equivalent(SpectralModel.unit_energy):
            raise ValueError
        
        self.index = float(index)
        self.E_peak = E_peak

    def E_cutoff(self):
        """Energy of the exponential cutoff"""
        return self.E_peak/(2+self.index)

    def _spectral_model(self, energy_normalised):
        """Evaluate the spectral model"""
        E_cutoff_normalised = (self.E_cutoff()/self.E0).to('')
        value = np.power(energy_normalised, self.index) * np.exp(- energy_normalised / E_cutoff_normalised)
        return value
    
    def __str__(self):
        string = f"Comptonized Model\n N0={self.N0:.3e}\n E0={self.E0}\n"
        string+= f" Index={self.index}\n"
        string+= f" Peak Energy={self.E_peak}\n Cutoff Energy={self.E_cutoff():.3e}\n"
        return string


# Examples

In [ ]:
# SOFT: Band 10 10000 -1.9 -3.7 699.9. FLUX=5 ph/s/cm2
soft = BandModel(lower_index=-1.9, upper_index=-3.7, E_break=230.0*u.keV)
soft_N0 = soft.estimate_N0_from_flux(E1=10*u.keV, E2=10*u.MeV, flux=5*u.Unit("s-1 cm-2"))
soft.set_N0(soft_N0)

display(soft)
display(f"Flux 10 keV - 10 MeV = {soft.get_flux(E1=10*u.keV, E2=10*u.MeV)}") # Sanity Check


In [ ]:
# MEDIUM: Band 10 10000 -1 -2.3 230. FLUX=5 ph/s/cm2
medium = BandModel(lower_index=-1.0, upper_index=-2.3, E_break=699.9*u.keV)
medium_N0 = medium.estimate_N0_from_flux(E1=10*u.keV, E2=10*u.MeV, flux=5*u.Unit("s-1 cm-2"))
medium.set_N0(medium_N0)

display(medium)
display(f"Flux 10 keV - 10 MeV = {medium.get_flux(E1=10*u.keV, E2=10*u.MeV)}") # Sanity Check

In [ ]:
# HARD: Compotonized 10 10000 -0.5 1500. FLUX=5 ph/s/cm2
hard = Comptonized(index=-0.5, E_peak=1500*u.keV)
hard_N0 = hard.estimate_N0_from_flux(E1=10*u.keV, E2=10*u.MeV, flux=5*u.Unit("s-1 cm-2"))
hard.set_N0(hard_N0)
display(hard)

display(f"Flux 10 keV - 10 MeV = {hard.get_flux(E1=10*u.keV, E2=10*u.MeV)}") # Sanity Check

### Plot

In [ ]:
energies = np.logspace(np.log10(10),np.log10(10000), num=51) * u.keV
# energies

Spectra as dNdE (ph/s/cm2/keV)

In [ ]:
# plt.rc('font'  , family="Times New Roman", style="normal", variant="normal",weight="normal")

FIGSIZE=(16,8)


fig, ax = plt.subplots(1, figsize=FIGSIZE, constrained_layout=True)
plt.setp(ax.spines.values(), linewidth=1.5)

x = energies

y = soft.get_dNdE(energies)
ax.plot(x, y, lw=3, label=f"Soft {soft}", color="r")

#y = soft_old.get_dNdE(energies)
#ax.plot(x, y, lw=3, label=f"Soft old {soft_old}", color="r",linestyle='--')

y = medium.get_dNdE(energies)
ax.plot(x, y, lw=3, label=f"Medium {medium}", color="b")

#y = medium_old.get_dNdE(energies)
#ax.plot(x, y, lw=3, label=f"Medium old {medium_old}", color="b",linestyle='--')

y = hard.get_dNdE(energies)
ax.plot(x, y, lw=3, label=f"Hard {hard}", color="g")

#y = hard_old.get_dNdE(energies)
#ax.plot(x, y, lw=3, label=f"Hard old {hard_old}", color="g",linestyle='--')

y = fermi_grb.get_dNdE(energies)
ax.plot(x, y, lw=3, label=f"Fermi {fermi_grb}", color="m")

ax.axvline(80, c='k')
ax.axvline(2000, c='k')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(f"Energy ({x.unit})",fontsize=15)
ax.set_ylabel(f"dNdE ({y.unit})",fontsize=15)
plt.legend(fontsize=15)
ax.grid(ls="--")
ax.set_title("Spectral Models")
ax.legend(bbox_to_anchor=(1.0,1.0))
plt.show()

In [ ]:
# plt.rc('font'  , family="Times New Roman", style="normal", variant="normal",weight="normal")

FIGSIZE=(12,8)


fig, ax = plt.subplots(1, figsize=FIGSIZE, constrained_layout=True)
plt.setp(ax.spines.values(), linewidth=1.5)

x = energies

y = soft.get_dNdE(energies)
ax.plot(x, y, lw=3, label=f"Soft Band Spectrum", color="r")

#y = soft_old.get_dNdE(energies)
#ax.plot(x, y, lw=3, label=f"Soft old {soft_old}", color="r",linestyle='--')

y = medium.get_dNdE(energies)
ax.plot(x, y, lw=3, label=f"Medium Band Spectrum", color="b")

#y = medium_old.get_dNdE(energies)
#ax.plot(x, y, lw=3, label=f"Medium old {medium_old}", color="b",linestyle='--')

y = hard.get_dNdE(energies)
ax.plot(x, y, lw=3, label=f"Hard Comptonized Spectrum", color="g")

#y = hard_old.get_dNdE(energies)
#ax.plot(x, y, lw=3, label=f"Hard old {hard_old}", color="g",linestyle='--')

ax.axvline(80, c='k')
ax.axvline(2000, c='k')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(f"Energy ({x.unit})",fontsize=22)
ax.set_ylabel(f"dNdE ({y.unit})",fontsize=22)
ax.tick_params(axis='both', labelsize=22) 
ax.grid(ls="--")
ax.set_title("")
ax.legend(fontsize=22,loc='upper right')
#plt.legend()
plt.show()

Spectra as SED (erg/s/cm2)

In [ ]:
fig, ax = plt.subplots(1, figsize=FIGSIZE, constrained_layout=True)
plt.setp(ax.spines.values(), linewidth=1.5)

x = energies
y = soft.get_E2dNdE(energies)
ax.plot(x, y, lw=3, label=f"Soft {soft}")

y = medium.get_E2dNdE(energies)
ax.plot(x, y, lw=3, label=f"Medium {medium}")

y = hard.get_E2dNdE(energies)
ax.plot(x, y, lw=3, label=f"Hard {hard}")

ax.axvline(80, c='k')
ax.axvline(10000, c='k')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(f"Energy ({x.unit})")
ax.set_ylabel(f"E2dNdE ({y.unit})")
ax.grid(ls="--")
ax.set_title("Spectral Models")
ax.legend(bbox_to_anchor=(1.0,1.0))
plt.show()

In [ ]:
# COSI VERSION
E0 = 300*u.keV

# SOFT
soft = BandModel(lower_index=-1.9, upper_index=-3.7, E_break=230*u.keV)
soft.set_E0(E0)
soft_N0 = soft.estimate_N0_from_flux(E1=10*u.keV, E2=10*u.MeV, flux=500*u.Unit("s-1 cm-2"))
soft.set_N0(soft_N0)
#print(f"Soft Model Flux  = {soft.get_flux(E1=10*u.keV, E2=10*u.MeV)}\n{soft}\n")

# MEDIUM
medium = BandModel(lower_index=-1.0, upper_index=-2.3, E_break=699.9*u.keV)
medium.set_E0(E0)
# medium_N0 = medium.estimate_N0_from_flux(E1=10*u.keV, E2=10*u.MeV, flux=200*u.Unit("s-1 cm-2"))
medium_N0 = soft.get_dNdE(E0)/np.exp(-E0/(699.9*u.keV))
medium.set_N0(medium_N0)
#print(f"Medium Model Flux = {medium.get_flux(E1=10*u.keV, E2=10*u.MeV)}\n{medium}\n")

# HARD
hard = Comptonized(index=-0.5, E_peak=1500*u.keV)
hard.set_E0(E0)
# hard_N0 = hard.estimate_N0_from_flux(E1=10*u.keV, E2=10*u.MeV, flux=100*u.Unit("s-1 cm-2"))
hard_N0 = soft.get_dNdE(E0)/np.exp(-E0*(2-0.5)/(1500*u.keV))
hard.set_N0(hard_N0)
#print(f"Hard Model Flux = {hard.get_flux(E1=10*u.keV, E2=10*u.MeV)}\n{hard}\n")

print(soft.get_dNdE(E0))
print(medium.get_dNdE(E0))
print(hard.get_dNdE(E0))


In [ ]:
# energies
energies = np.logspace(np.log10(10),np.log10(10000), num=51) * u.keV
##########################

fig, axs = plt.subplots(1, 2, figsize=(20,4), constrained_layout=True)

########### dNdE
axs[0].plot(energies, soft.get_dNdE(energies)  , lw=2, ls="-", c='r', label=f"Soft Band Spectrum")
axs[0].plot(energies, medium.get_dNdE(energies), lw=2, ls="-", c='b', label=f"Medium band Spectrum")
axs[0].plot(energies, hard.get_dNdE(energies)  , lw=2, ls="-", c='g', label=f"Hard Comptonized Spectrum")
axs[0].set_ylabel(f"dN/dE [{soft.get_dNdE(energies).unit}]")
###########

########### E2 dNdE
axs[1].plot(energies, soft.get_E2dNdE(energies)  , lw=2, ls="-", c='r', label=f"Soft Band Spectrum")
axs[1].plot(energies, medium.get_E2dNdE(energies), lw=2, ls="-", c='b', label=f"Medium band Spectrum")
axs[1].plot(energies, hard.get_E2dNdE(energies)  , lw=2, ls="-", c='g', label=f"Hard Comptonized Spectrum")
axs[1].set_ylabel(f"E2 dN/dE [{soft.get_E2dNdE(energies).unit}]")
###########


for ax in axs:
    # plt.setp(ax.spines.values(), linewidth=1.5)
    ax.axvline(80, c='k')
    ax.axvline(2000, c='k')
    ax.axvline(300,ls="--", c='k')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(f"Energy [{energies.unit}]")
    ax.grid(ls=":")
    ax.set_title("Spectral Models")
    
axs[0].legend()    
plt.show()

In [ ]:
FIGSIZE=(12,8)


fig, ax = plt.subplots(1, figsize=FIGSIZE, constrained_layout=True)
plt.setp(ax.spines.values(), linewidth=1.5)

########### dNdE
ax.plot(energies, soft.get_dNdE(energies)  , lw=3, ls="-", c='r', label=f"Soft Band Spectrum")
ax.plot(energies, medium.get_dNdE(energies), lw=3, ls="-", c='b', label=f"Medium band Spectrum")
ax.plot(energies, hard.get_dNdE(energies)  , lw=3, ls="-", c='g', label=f"Hard Comptonized Spectrum")
ax.set_ylabel(f"dN/dE [{soft.get_dNdE(energies).unit}]")
###########

# plt.setp(ax.spines.values(), linewidth=1.5)
ax.axvline(80, c='k')
ax.axvline(2000, c='k')
ax.axvline(300,ls="--", c='k')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(f"Energy [{energies.unit}]")
ax.grid(ls=":")
ax.set_xlabel(f"Energy ({x.unit})",fontsize=22)
ax.set_ylabel(f"dNdE ({y.unit})",fontsize=22)
ax.tick_params(axis='both', labelsize=22) 

ax.legend(fontsize=22,loc='upper right') 
plt.show()